## DMEPOS-Supplier Data Carpentry

The purpose of this notebook is to clean the DMEPOS-by Supplier dataset for exploration and ML modeling.

The cleaning steps involved:
- Assessed data structure, quality, and consistency
- Identified missing and suppressed values
- Removed non-essential columns
- Retained core supplier, claims, beneficiary, charge, and risk metrics
- Added a Year column
- Reordered columns for consistency
- Combined all three years into a single dataset
- Cleaned and standardized string columns
- Mapped suppression indicators to 'y' and 'n'
- Replaced numeric suppressed values with a placeholder

End result: 

Saved the cleaned dataset to a shared team directory: /dsa/groups/casestudycf25/team02/DMEPOS_cleaned_suplr.csv

## Loading the data and quick exploration

In [1]:
# Loading Libraries
import re
import pandas as pd
from pathlib import Path

Using Path function to verify if the files exist by simply listing all files and folders in the specified directory, printing their names.

In [2]:
for f in Path("/dsa/groups/casestudycf25/team02/").iterdir():
    print(f.name)

DMEPOS_suplr_2021.csv
Medicare_Monthly_Enrollment_Jun_2025.csv
DMEPOS_suplr_2022.csv
DMEPOS_suplr_2023.csv
DMEPOS_Supplier_Service_2021.csv
DMEPOS_Supplier_Service_2022.csv
DMEPOS_Supplier_Service_2023.csv
LEIE_OIG_Exclusion_List.csv
mup_dme_ry25_p05_v10_dy21_rfrhpr.csv
mup_dme_ry25_p05_v10_dy21_rfrr.csv
mup_dme_ry25_p05_v10_dy22_rfrhpr.csv
mup_dme_ry25_p05_v10_dy22_rfrr.csv
mup_dme_ry25_p05_v10_dy23_rfrhpr.csv
mup_dme_ry25_p05_v10_dy23_rfrr.csv
RUCA2010zipcode.csv
Taxonomy Code List Dec 2023.csv
leie_with_null_npi_clean.csv
leie_with_valid_npi_clean.csv
medicare_enrollment_clean.csv
CMS_General_Payments_2021_2023_raw.csv
casestudycf25t02.sqlite.db
DMEPOS_Bene_only_suplr.csv
DMEPOS_rfrhpr_clean.csv
DMEPOS_rfrr_clean.csv
DMEPOS_RefPro_Ser_clean_02.csv
DMEPOS_cleaned_serv.csv
DMEPOS_cleaned_grouping.csv
OWNRSHP_PGYR2021_2023.csv
ownership_payment_clean.csv
DMEPOS_rfrhpr_clean_labeled.csv
DMEPOS_rfrr_clean_labeled.csv
general_payments_manufacturers_clean.csv
general_payments_providers_cle

Loading the 3 CSV files into separate pandas dataframes.

In [4]:
df21 = pd.read_csv("/dsa/groups/casestudycf25/team02/DMEPOS_suplr_2021.csv",na_values=["NA"])
df22 = pd.read_csv("/dsa/groups/casestudycf25/team02/DMEPOS_suplr_2022.csv",na_values=["NA"])
df23 = pd.read_csv("/dsa/groups/casestudycf25/team02/DMEPOS_suplr_2023.csv",na_values=["NA"])

Taking a quick peak at the dataframes to see if everything is loaded.  I can also see the structure of the datasets.

In [5]:
df21.head()

,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,Suplr_Prvdr_State_Abrvtn,...,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre
0,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,0.271552,0.120690,0.741379,0.650862,0.215517,0.189655,NaN,0.676724,0.094828,0.982410
1,1003002254,Walgreen Co.,NaN,NaN,NaN,O,5104 Bobby Hicks Hwy,NaN,Gray,TN,...,0.854545,0.236364,0.818182,0.836364,0.363636,NaN,NaN,0.272727,NaN,1.500596
2,1003004904,Texas Road Old Bridge Llc,NaN,NaN,NaN,O,1183 Englishtown Rd,NaN,Old Bridge,NJ,...,1.190476,NaN,1.095238,1.190476,0.809524,NaN,NaN,0.571429,NaN,2.327878
3,1003004938,"Cvs State Capital, L.L.C.",NaN,NaN,NaN,O,446 Sabattus St,NaN,Lewiston,ME,...,0.757576,0.242424,0.757576,0.787879,0.318182,NaN,NaN,0.363636,NaN,1.693562
4,1003007386,"The Giant Company, Llc",NaN,NaN,NaN,O,925 Norland Ave,NaN,Chambersburg,PA,...,1.000000,NaN,0.947368,0.973684,NaN,NaN,0.0,0.342105,NaN,1.188676


In [6]:
df22.head()

,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,Suplr_Prvdr_State_Abrvtn,...,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre
0,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,0.271111,0.080000,0.711111,0.648889,0.275556,0.164444,NaN,0.648889,NaN,0.886236
1,1003002254,Walgreen Co.,NaN,NaN,NaN,O,5104 Bobby Hicks Hwy,NaN,Gray,TN,...,0.836364,0.290909,0.818182,0.854545,0.400000,NaN,0.0,0.400000,NaN,1.678905
2,1003004904,Texas Road Old Bridge Llc,NaN,NaN,NaN,O,1183 Englishtown Rd,NaN,Old Bridge,NJ,...,0.809524,NaN,0.857143,0.904762,NaN,NaN,NaN,0.666667,NaN,2.357026
3,1003004938,"Cvs State Capital, L.L.C.",NaN,NaN,NaN,O,446 Sabattus St,NaN,Lewiston,ME,...,0.755102,NaN,0.714286,0.734694,NaN,0.000000,NaN,0.387755,NaN,1.175749
4,1003007386,"The Giant Company, Llc",NaN,NaN,NaN,O,925 Norland Ave,NaN,Chambersburg,PA,...,0.833333,0.229167,0.854167,0.937500,0.312500,NaN,0.0,0.416667,NaN,1.373510


In [7]:
df23.head()

,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,Suplr_Prvdr_State_Abrvtn,...,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre
0,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,0.195980,0.105528,0.708543,0.673367,0.221106,0.20603,NaN,0.663317,NaN,1.016809
1,1003002254,Walgreen Co,NaN,NaN,NaN,O,5104 Bobby Hicks Hwy,NaN,Gray,TN,...,0.740000,0.240000,0.740000,0.860000,0.360000,NaN,0.0,0.340000,NaN,1.873065
2,1003004904,Texas Road Old Bridge Llc,NaN,NaN,NaN,O,1183 Englishtown Rd,NaN,Old Bridge,NJ,...,0.933333,NaN,1.000000,1.000000,NaN,NaN,0.0,NaN,0.0,1.091813
3,1003004938,"Cvs State Capital, L.L.C.",NaN,NaN,NaN,O,446 Sabattus St,NaN,Lewiston,ME,...,0.800000,NaN,0.771429,0.828571,NaN,NaN,0.0,0.400000,NaN,1.546691
4,1003007386,"The Giant Company, Llc",NaN,NaN,NaN,O,925 Norland Ave,NaN,Chambersburg,PA,...,0.978723,NaN,0.936170,1.000000,0.425532,NaN,NaN,0.510638,NaN,1.329664


In [8]:
# Looking at the column names....

df23.columns.tolist()

['Suplr_NPI',
 'Suplr_Prvdr_Last_Name_Org',
 'Suplr_Prvdr_First_Name',
 'Suplr_Prvdr_MI',
 'Suplr_Prvdr_Crdntls',
 'Suplr_Prvdr_Ent_Cd',
 'Suplr_Prvdr_St1',
 'Suplr_Prvdr_St2',
 'Suplr_Prvdr_City',
 'Suplr_Prvdr_State_Abrvtn',
 'Suplr_Prvdr_State_FIPS',
 'Suplr_Prvdr_Zip5',
 'Suplr_Prvdr_RUCA',
 'Suplr_Prvdr_RUCA_Desc',
 'Suplr_Prvdr_Cntry',
 'Suplr_Prvdr_Spclty_Desc',
 'Suplr_Prvdr_Spclty_Srce',
 'Tot_Suplr_HCPCS_Cds',
 'Tot_Suplr_Benes',
 'Tot_Suplr_Clms',
 'Tot_Suplr_Srvcs',
 'Suplr_Sbmtd_Chrgs',
 'Suplr_Mdcr_Alowd_Amt',
 'Suplr_Mdcr_Pymt_Amt',
 'Suplr_Mdcr_Stdzd_Pymt_Amt',
 'DME_Sprsn_Ind',
 'DME_Tot_Suplr_HCPCS_Cds',
 'DME_Tot_Suplr_Benes',
 'DME_Tot_Suplr_Clms',
 'DME_Tot_Suplr_Srvcs',
 'DME_Suplr_Sbmtd_Chrgs',
 'DME_Suplr_Mdcr_Alowd_Amt',
 'DME_Suplr_Mdcr_Pymt_Amt',
 'DME_Suplr_Mdcr_Stdzd_Pymt_Amt',
 'POS_Sprsn_Ind',
 'POS_Tot_Suplr_HCPCS_Cds',
 'POS_Tot_Suplr_Benes',
 'POS_Tot_Suplr_Clms',
 'POS_Tot_Suplr_Srvcs',
 'POS_Suplr_Sbmtd_Chrgs',
 'POS_Suplr_Mdcr_Alowd_Amt',
 'POS_Supl

These are the core attributes I will be focusing on (for now):
- Year
- Suplr_NPI
- Suplr_Prvdr_Last_Name_Org
- Suplr_Prvdr_First_Name
- Suplr_Prvdr_MI
- Suplr_Prvdr_Crdntls
- Suplr_Prvdr_Ent_Cd
- Suplr_Prvdr_St1
- Suplr_Prvdr_St2
- Suplr_Prvdr_City
- Suplr_Prvdr_State_Abrvtn
- Suplr_Prvdr_State_FIPS
- Suplr_Prvdr_Zip5
- Suplr_Prvdr_Cntry
- Suplr_Prvdr_Spclty_Desc
- Suplr_Prvdr_Spclty_Srce
- Tot_Suplr_HCPCS_Cds
- Tot_Suplr_Benes
- Tot_Suplr_Clms
- Tot_Suplr_Srvcs
- Suplr_Sbmtd_Chrgs
- Suplr_Mdcr_Alowd_Amt
- Suplr_Mdcr_Pymt_Amt
- Suplr_Mdcr_Stdzd_Pymt_Amt
- DME_Sprsn_Ind
- DME_Tot_Suplr_HCPCS_Cds
- DME_Tot_Suplr_Benes
- DME_Tot_Suplr_Clms
- DME_Tot_Suplr_Srvcs
- DME_Suplr_Sbmtd_Chrgs
- DME_Suplr_Mdcr_Alowd_Amt
- DME_Suplr_Mdcr_Pymt_Amt
- DME_Suplr_Mdcr_Stdzd_Pymt_Amt
- POS_Sprsn_Ind
- POS_Tot_Suplr_HCPCS_Cds
- POS_Tot_Suplr_Benes
- POS_Tot_Suplr_Clms
- POS_Tot_Suplr_Srvcs
- POS_Suplr_Sbmtd_Chrgs
- POS_Suplr_Mdcr_Alowd_Amt
- POS_Suplr_Mdcr_Pymt_Amt
- POS_Suplr_Mdcr_Stdzd_Pymt_Amt
- Bene_Avg_Risk_Scre

Looking at the data types of the attributes.

In [9]:
df23.dtypes

Suplr_NPI                           int64
Suplr_Prvdr_Last_Name_Org          object
Suplr_Prvdr_First_Name             object
Suplr_Prvdr_MI                     object
Suplr_Prvdr_Crdntls                object
                                   ...   
Bene_CC_PH_Osteoporosis_V2_Pct    float64
Bene_CC_PH_Parkinson_V2_Pct       float64
Bene_CC_PH_Arthritis_V2_Pct       float64
Bene_CC_PH_Stroke_TIA_V2_Pct      float64
Bene_Avg_Risk_Scre                float64
Length: 93, dtype: object

## Carpentry

The below code adds a Year column to each yearly dataset, reorders columns so that Year comes first, and then combines all three years into a single DataFrame. The result is one dataset with a consistent structure across 2021, 2022, and 2023, that we can use for analysis.

In [10]:
# Adding a 'Year' column
df21['Year'] = 2021
df22['Year'] = 2022
df23['Year'] = 2023

# Reordering the columns
df21 = df21[['Year'] + [c for c in df21.columns if c != 'Year']]
df22 = df22[['Year'] + [c for c in df22.columns if c != 'Year']]
df23 = df23[['Year'] + [c for c in df23.columns if c != 'Year']]

# Combining all three datasets
df = pd.concat([df21, df22, df23], ignore_index=True)

In [11]:
# checking...
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre
0,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,0.271552,0.120690,0.741379,0.650862,0.215517,0.189655,NaN,0.676724,0.094828,0.982410
1,2021,1003002254,Walgreen Co.,NaN,NaN,NaN,O,5104 Bobby Hicks Hwy,NaN,Gray,...,0.854545,0.236364,0.818182,0.836364,0.363636,NaN,NaN,0.272727,NaN,1.500596
2,2021,1003004904,Texas Road Old Bridge Llc,NaN,NaN,NaN,O,1183 Englishtown Rd,NaN,Old Bridge,...,1.190476,NaN,1.095238,1.190476,0.809524,NaN,NaN,0.571429,NaN,2.327878
3,2021,1003004938,"Cvs State Capital, L.L.C.",NaN,NaN,NaN,O,446 Sabattus St,NaN,Lewiston,...,0.757576,0.242424,0.757576,0.787879,0.318182,NaN,NaN,0.363636,NaN,1.693562
4,2021,1003007386,"The Giant Company, Llc",NaN,NaN,NaN,O,925 Norland Ave,NaN,Chambersburg,...,1.000000,NaN,0.947368,0.973684,NaN,NaN,0.0,0.342105,NaN,1.188676


In [12]:
# Checking
df.tail()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre
198616,2023,1992989289,"Carl G Purvis, Dpm Pa",NaN,NaN,NaN,O,3301 Sunset Ave,NaN,Rocky Mount,...,0.808036,0.241071,0.919643,0.915179,0.375000,0.098214,NaN,0.566964,0.129464,1.608311
198617,2023,1992989537,Meru Pharmacy Inc,NaN,NaN,NaN,O,2 Park Ave,NaN,Yonkers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.829333
198618,2023,1992989552,"Harris Teeter, Llc",NaN,NaN,NaN,O,1140 Green Level Church Rd,NaN,Cary,...,0.900000,NaN,0.900000,0.800000,NaN,NaN,NaN,NaN,0.000000,1.185842
198619,2023,1992989909,Joseph P Gabryszewski,NaN,NaN,NaN,O,3 Kirchner Ave,NaN,Hyde Park,...,0.935484,NaN,0.935484,0.870968,0.354839,NaN,NaN,0.548387,NaN,1.866097
198620,2023,1992999106,"Northwest Indiana Eye Associates, Pc",NaN,NaN,NaN,O,297 W. Franciscan Dr.,Suite 101,Crown Point,...,0.350649,NaN,0.779221,0.831169,0.337662,0.155844,NaN,0.441558,NaN,1.027333


Looking at the attributes of the combined dataframe.

For the time being, only specific columns will be retained for the time being to be used for analysis.Beneficiary demographics will be looked into at another time.

The removed attributes are as follows:
- Drug_Sprsn_Ind
- Drug_Tot_Suplr_HCPCS_Cds
- Drug_Tot_Suplr_Benes
- Drug_Tot_Suplr_Clms
- Drug_Tot_Suplr_Srvcs
- Drug_Suplr_Sbmtd_Chrgs
- Drug_Suplr_Mdcr_Alowd_Amt
- Drug_Suplr_Mdcr_Pymt_Amt
- Drug_Suplr_Mdcr_Stdzd_Pymt_Amt
- Bene_Avg_Age
- Bene_Age_LT_65_Cnt
- Bene_Age_65_74_Cnt
- Bene_Age_75_84_Cnt
- Bene_Age_GT_84_Cnt
- Bene_Feml_Cnt
- Bene_Male_Cnt
- Bene_Race_Wht_Cnt
- Bene_Race_Black_Cnt
- Bene_Race_Api_Cnt
- Bene_Race_Hspnc_Cnt
- Bene_Race_Natind_Cnt
- Bene_Race_Othr_Cnt
- Bene_Ndual_Cnt
- Bene_Dual_Cnt
- Bene_CC_BH_ADHD_OthCD_V1_Pct
- Bene_CC_BH_Alcohol_Drug_V1_Pct
- Bene_CC_BH_Tobacco_V1_Pct
- Bene_CC_BH_Alz_NonAlzdem_V2_Pct
- Bene_CC_BH_Anxiety_V1_Pct
- Bene_CC_BH_Bipolar_V1_Pct
- Bene_CC_BH_Mood_V2_Pct
- Bene_CC_BH_Depress_V1_Pct
- Bene_CC_BH_PD_V1_Pct
- Bene_CC_BH_PTSD_V1_Pct
- Bene_CC_BH_Schizo_OthPsy_V1_Pct
- Bene_CC_PH_Asthma_V2_Pct
- Bene_CC_PH_Afib_V2_Pct
- Bene_CC_PH_Cancer6_V2_Pct
- Bene_CC_PH_CKD_V2_Pct
- Bene_CC_PH_COPD_V2_Pct
- Bene_CC_PH_Diabetes_V2_Pct
- Bene_CC_PH_HF_NonIHD_V2_Pct
- Bene_CC_PH_Hyperlipidemia_V2_Pct
- Bene_CC_PH_Hypertension_V2_Pct
- Bene_CC_PH_IschemicHeart_V2_Pct
- Bene_CC_PH_Osteoporosis_V2_Pct
- Bene_CC_PH_Parkinson_V2_Pct
- Bene_CC_PH_Arthritis_V2_Pct
- Bene_CC_PH_Stroke_TIA_V2_Pct
- Suplr_Prvdr_RUCA
- Suplr_Prvdr_RUCA_Desc

Drug-related, beneficiary demographic, clinical condition, and provider RUCA attributes will be removed, leaving only core DME/POS metrics and essential provider identifiers for analysis.

In [13]:
# Columns to be removed
rem_att = ['Drug_Sprsn_Ind', 'Drug_Tot_Suplr_HCPCS_Cds', 'Drug_Tot_Suplr_Benes',
           'Drug_Tot_Suplr_Clms', 'Drug_Tot_Suplr_Srvcs', 'Drug_Suplr_Sbmtd_Chrgs',
           'Drug_Suplr_Mdcr_Alowd_Amt', 'Drug_Suplr_Mdcr_Pymt_Amt',
           'Drug_Suplr_Mdcr_Stdzd_Pymt_Amt', 'Bene_Avg_Age', 'Bene_Age_LT_65_Cnt',
           'Bene_Age_65_74_Cnt', 'Bene_Age_75_84_Cnt', 'Bene_Age_GT_84_Cnt',
           'Bene_Feml_Cnt', 'Bene_Male_Cnt', 'Bene_Race_Wht_Cnt', 'Bene_Race_Black_Cnt',
           'Bene_Race_Api_Cnt', 'Bene_Race_Hspnc_Cnt', 'Bene_Race_Natind_Cnt',
           'Bene_Race_Othr_Cnt', 'Bene_Ndual_Cnt', 'Bene_Dual_Cnt',
           'Bene_CC_BH_ADHD_OthCD_V1_Pct', 'Bene_CC_BH_Alcohol_Drug_V1_Pct',
           'Bene_CC_BH_Tobacco_V1_Pct', 'Bene_CC_BH_Alz_NonAlzdem_V2_Pct',
           'Bene_CC_BH_Anxiety_V1_Pct', 'Bene_CC_BH_Bipolar_V1_Pct',
           'Bene_CC_BH_Mood_V2_Pct', 'Bene_CC_BH_Depress_V1_Pct',
           'Bene_CC_BH_PD_V1_Pct', 'Bene_CC_BH_PTSD_V1_Pct',
           'Bene_CC_BH_Schizo_OthPsy_V1_Pct', 'Bene_CC_PH_Asthma_V2_Pct',
           'Bene_CC_PH_Afib_V2_Pct', 'Bene_CC_PH_Cancer6_V2_Pct',
           'Bene_CC_PH_CKD_V2_Pct', 'Bene_CC_PH_COPD_V2_Pct', 'Bene_CC_PH_Diabetes_V2_Pct',
           'Bene_CC_PH_HF_NonIHD_V2_Pct', 'Bene_CC_PH_Hyperlipidemia_V2_Pct',
           'Bene_CC_PH_Hypertension_V2_Pct', 'Bene_CC_PH_IschemicHeart_V2_Pct',
           'Bene_CC_PH_Osteoporosis_V2_Pct', 'Bene_CC_PH_Parkinson_V2_Pct',
           'Bene_CC_PH_Arthritis_V2_Pct', 'Bene_CC_PH_Stroke_TIA_V2_Pct',
           'Suplr_Prvdr_RUCA', 'Suplr_Prvdr_RUCA_Desc']

# Drop the columns
df = df.drop(columns=rem_att)

# Checking....
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,POS_Sprsn_Ind,POS_Tot_Suplr_HCPCS_Cds,POS_Tot_Suplr_Benes,POS_Tot_Suplr_Clms,POS_Tot_Suplr_Srvcs,POS_Suplr_Sbmtd_Chrgs,POS_Suplr_Mdcr_Alowd_Amt,POS_Suplr_Mdcr_Pymt_Amt,POS_Suplr_Mdcr_Stdzd_Pymt_Amt,Bene_Avg_Risk_Scre
0,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,NaN,16.0,232.0,316.0,366.0,89753.0,69895.17,54863.24,53816.15,0.982410
1,2021,1003002254,Walgreen Co.,NaN,NaN,NaN,O,5104 Bobby Hicks Hwy,NaN,Gray,...,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.500596
2,2021,1003004904,Texas Road Old Bridge Llc,NaN,NaN,NaN,O,1183 Englishtown Rd,NaN,Old Bridge,...,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,2.327878
3,2021,1003004938,"Cvs State Capital, L.L.C.",NaN,NaN,NaN,O,446 Sabattus St,NaN,Lewiston,...,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.693562
4,2021,1003007386,"The Giant Company, Llc",NaN,NaN,NaN,O,925 Norland Ave,NaN,Chambersburg,...,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.188676


Printing the size of the dataset: the number of rows (data entries) and the number of columns (attributes). It gives a quick overview of how much data is available and how many features can be analyzed.

In [14]:
print(f"No. of data entries: {df.shape[0]}")
print(f"No. of attributes: {df.shape[1]}")

No. of data entries: 198621
No. of attributes: 43


It seems that there are quite a few missing values in the string, its best to replace them with 'na'.
It ensures that models (during the model phase) can process all rows without errors. Additionally, replacing nulls with a placeholder like 'na' preserves data, prevents row loss.

In [15]:
# Identify string columns, but excluding indicators
string_cols = [c for c in df.select_dtypes(include='object').columns
               if c not in ['DME_Sprsn_Ind', 'POS_Sprsn_Ind']]

In [16]:
# Clean string columns
for c in string_cols:
    # Convert to lowercase
    df[c] = df[c].str.lower()

    # Remove special characters minus letters, numbers, and spaces
    df[c] = df[c].str.replace(r"[^A-Za-z0-9\s]", "", regex=True)

    # Replace one or more spaces with underscore
    df[c] = df[c].str.replace(r"\s+", "_", regex=True)

    # Fill null values with 'na'
    df[c] = df[c].fillna("na")

# Checking......
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,POS_Sprsn_Ind,POS_Tot_Suplr_HCPCS_Cds,POS_Tot_Suplr_Benes,POS_Tot_Suplr_Clms,POS_Tot_Suplr_Srvcs,POS_Suplr_Sbmtd_Chrgs,POS_Suplr_Mdcr_Alowd_Amt,POS_Suplr_Mdcr_Pymt_Amt,POS_Suplr_Mdcr_Stdzd_Pymt_Amt,Bene_Avg_Risk_Scre
0,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,NaN,16.0,232.0,316.0,366.0,89753.0,69895.17,54863.24,53816.15,0.982410
1,2021,1003002254,walgreen_co,na,na,na,o,5104_bobby_hicks_hwy,na,gray,...,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.500596
2,2021,1003004904,texas_road_old_bridge_llc,na,na,na,o,1183_englishtown_rd,na,old_bridge,...,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,2.327878
3,2021,1003004938,cvs_state_capital_llc,na,na,na,o,446_sabattus_st,na,lewiston,...,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.693562
4,2021,1003007386,the_giant_company_llc,na,na,na,o,925_norland_ave,na,chambersburg,...,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.188676


The below code check for empty values in the dataframe.

In [17]:
# Count of null values in each column
null_counts = df.isnull().sum()

# Display columns with any null values
null_counts[null_counts > 0]

Tot_Suplr_Benes                   14391
DME_Sprsn_Ind                    188204
DME_Tot_Suplr_HCPCS_Cds           10417
DME_Tot_Suplr_Benes               25712
DME_Tot_Suplr_Clms                10417
DME_Tot_Suplr_Srvcs               10417
DME_Suplr_Sbmtd_Chrgs             10417
DME_Suplr_Mdcr_Alowd_Amt          10417
DME_Suplr_Mdcr_Pymt_Amt           10417
DME_Suplr_Mdcr_Stdzd_Pymt_Amt     10417
POS_Sprsn_Ind                    159285
POS_Tot_Suplr_HCPCS_Cds           39336
POS_Tot_Suplr_Benes               52679
POS_Tot_Suplr_Clms                39336
POS_Tot_Suplr_Srvcs               39336
POS_Suplr_Sbmtd_Chrgs             39336
POS_Suplr_Mdcr_Alowd_Amt          39336
POS_Suplr_Mdcr_Pymt_Amt           39336
POS_Suplr_Mdcr_Stdzd_Pymt_Amt     39336
Bene_Avg_Risk_Scre                   35
dtype: int64

Aside DME_Sprsn_Ind and POS_Sprsn_Ind the remaining attributes are numerics. The empty spaces represents suppression due to suppliers have a beneficiary count less than 10.

So, first I will set up the indicators to show which rows are suppressed and which are not and replace the empty values with a placeholder like "5". The placeholder will indicate suppression without implying zero.

In [18]:
# Columns that indicate suppression
sup_cols = ['DME_Sprsn_Ind', 'POS_Sprsn_Ind']

# Replace nulls or empty spaces with 'n' (no suppression)
for col in sup_cols:
    df[col] = df[col].replace(["", " "], "n")
    df[col] = df[col].fillna("n")

# Replace suppressed indicators (*, #, with spaces) with 'y'
for col in sup_cols:
    df[col] = df[col].replace(["*", "#", " *", " #", "* ", "# "], "y")

In [19]:
# Find all numeric attributes
num_cols = df.select_dtypes(include='number').columns.tolist()

# Replace nulls with 5
df[num_cols] = df[num_cols].fillna(5)

In [20]:
# Checking
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,POS_Sprsn_Ind,POS_Tot_Suplr_HCPCS_Cds,POS_Tot_Suplr_Benes,POS_Tot_Suplr_Clms,POS_Tot_Suplr_Srvcs,POS_Suplr_Sbmtd_Chrgs,POS_Suplr_Mdcr_Alowd_Amt,POS_Suplr_Mdcr_Pymt_Amt,POS_Suplr_Mdcr_Stdzd_Pymt_Amt,Bene_Avg_Risk_Scre
0,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,n,16.0,232.0,316.0,366.0,89753.0,69895.17,54863.24,53816.15,0.982410
1,2021,1003002254,walgreen_co,na,na,na,o,5104_bobby_hicks_hwy,na,gray,...,n,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.500596
2,2021,1003004904,texas_road_old_bridge_llc,na,na,na,o,1183_englishtown_rd,na,old_bridge,...,n,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,2.327878
3,2021,1003004938,cvs_state_capital_llc,na,na,na,o,446_sabattus_st,na,lewiston,...,n,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.693562
4,2021,1003007386,the_giant_company_llc,na,na,na,o,925_norland_ave,na,chambersburg,...,n,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.188676


Verifying that there aren't any more empty values.

In [21]:
# Count of null values in each column
null_counts = df.isnull().sum()

# Display columns with any null values
null_counts[null_counts > 0]

Series([], dtype: int64)

Great, there is no empty values anymore. I can now move to saving this dataframe as a cleaned CSV and conducting EDA or modeling in the future.

In [22]:
# Quick check...
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,POS_Sprsn_Ind,POS_Tot_Suplr_HCPCS_Cds,POS_Tot_Suplr_Benes,POS_Tot_Suplr_Clms,POS_Tot_Suplr_Srvcs,POS_Suplr_Sbmtd_Chrgs,POS_Suplr_Mdcr_Alowd_Amt,POS_Suplr_Mdcr_Pymt_Amt,POS_Suplr_Mdcr_Stdzd_Pymt_Amt,Bene_Avg_Risk_Scre
0,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,n,16.0,232.0,316.0,366.0,89753.0,69895.17,54863.24,53816.15,0.982410
1,2021,1003002254,walgreen_co,na,na,na,o,5104_bobby_hicks_hwy,na,gray,...,n,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.500596
2,2021,1003004904,texas_road_old_bridge_llc,na,na,na,o,1183_englishtown_rd,na,old_bridge,...,n,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,2.327878
3,2021,1003004938,cvs_state_capital_llc,na,na,na,o,446_sabattus_st,na,lewiston,...,n,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.693562
4,2021,1003007386,the_giant_company_llc,na,na,na,o,925_norland_ave,na,chambersburg,...,n,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.188676


In [23]:
# Looking at column names
df.columns.tolist()

['Year',
 'Suplr_NPI',
 'Suplr_Prvdr_Last_Name_Org',
 'Suplr_Prvdr_First_Name',
 'Suplr_Prvdr_MI',
 'Suplr_Prvdr_Crdntls',
 'Suplr_Prvdr_Ent_Cd',
 'Suplr_Prvdr_St1',
 'Suplr_Prvdr_St2',
 'Suplr_Prvdr_City',
 'Suplr_Prvdr_State_Abrvtn',
 'Suplr_Prvdr_State_FIPS',
 'Suplr_Prvdr_Zip5',
 'Suplr_Prvdr_Cntry',
 'Suplr_Prvdr_Spclty_Desc',
 'Suplr_Prvdr_Spclty_Srce',
 'Tot_Suplr_HCPCS_Cds',
 'Tot_Suplr_Benes',
 'Tot_Suplr_Clms',
 'Tot_Suplr_Srvcs',
 'Suplr_Sbmtd_Chrgs',
 'Suplr_Mdcr_Alowd_Amt',
 'Suplr_Mdcr_Pymt_Amt',
 'Suplr_Mdcr_Stdzd_Pymt_Amt',
 'DME_Sprsn_Ind',
 'DME_Tot_Suplr_HCPCS_Cds',
 'DME_Tot_Suplr_Benes',
 'DME_Tot_Suplr_Clms',
 'DME_Tot_Suplr_Srvcs',
 'DME_Suplr_Sbmtd_Chrgs',
 'DME_Suplr_Mdcr_Alowd_Amt',
 'DME_Suplr_Mdcr_Pymt_Amt',
 'DME_Suplr_Mdcr_Stdzd_Pymt_Amt',
 'POS_Sprsn_Ind',
 'POS_Tot_Suplr_HCPCS_Cds',
 'POS_Tot_Suplr_Benes',
 'POS_Tot_Suplr_Clms',
 'POS_Tot_Suplr_Srvcs',
 'POS_Suplr_Sbmtd_Chrgs',
 'POS_Suplr_Mdcr_Alowd_Amt',
 'POS_Suplr_Mdcr_Pymt_Amt',
 'POS_Suplr_Mdcr_Std

In [24]:
# Save the cleaned dataset to CSV
df.to_csv("/dsa/groups/casestudycf25/team02/DMEPOS_cleaned_suplr.csv", index=False)